## Import the Libraries

In [ ]:
import os
import torch
import kornia as Ko
import cv2
import random
import time
import math
import torch.nn as nn
import torch.optim as optim
from mpl_toolkits.mplot3d import Axes3D
from torch.utils.data import Dataset, DataLoader, random_split
import numpy as np
from natsort import natsorted
from scipy.spatial.transform import Rotation as R_scipy
import matplotlib.pyplot as plt
import torch.nn.functional as F
from torchvision.models import resnet50
from scipy.spatial.transform import Rotation as R

## Image Preprocess, keypoint Filter and Extract Intrinsics

In [ ]:
def image_preprocess(IMG_PATH):
    all_imgs = []
    tensor_imgs = []
    for i in range(len(IMG_PATH)):
        img = cv2.imread(IMG_PATH[i],0)
        #img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img_tensor = Ko.image_to_tensor(img, False).float() / 255.0
        all_imgs.append(img)
        tensor_imgs.append(img_tensor)
    return all_imgs,tensor_imgs

def norm_and_filter(key1,key2,cam_K):

    k1_h = np.hstack([key1, np.ones((key1.shape[0], 1))])
    k2_h = np.hstack([key2, np.ones((key2.shape[0], 1))])

    k1_n = np.array([cam_K@k1_h[i] for i in range(len(k1_h))])
    k2_n = np.array([cam_K@k2_h[i] for i in range(len(k2_h))])

    return k1_n,k2_n

def extract_intrinsics(calib_path, cam='P0'):
    with open(calib_path, 'r') as f:
        lines = f.readlines()
    for line in lines:
        if line.startswith(cam + ':'):
            values = list(map(float, line.strip().split()[1:]))
            P = np.array(values).reshape(3, 4)
            K = P[:, :3]
            return K
    raise ValueError(f"{cam} not found in calibration file")

## Parallel LoFTR processing

In [ ]:
def pad_to_divisible(img_tensor, divisor=8):
    if img_tensor.dim() == 4:
        h, w = img_tensor.shape[2], img_tensor.shape[3]
    else:
        h, w = img_tensor.shape[1], img_tensor.shape[2]
    pad_h = (divisor - h % divisor) % divisor
    pad_w = (divisor - w % divisor) % divisor
    if pad_h == 0 and pad_w == 0: return img_tensor, (h, w)
    return F.pad(img_tensor, (0, pad_w, 0, pad_h), value=0), (h, w)

def LOFTR_matcher_manual_split(tens_images, batch_size=8):
    num_gpus = torch.cuda.device_count()
    print(f"Running Manual Multi-GPU on {num_gpus} GPUs")

    # 1. Load Model on GPU 0
    matcher0 = Ko.feature.LoFTR(pretrained='outdoor').to('cuda:0')
    matcher0.eval()

    # 2. Load Model on GPU 1 (if available)
    matcher1 = None
    if num_gpus > 1:
        matcher1 = Ko.feature.LoFTR(pretrained='outdoor').to('cuda:1')
        matcher1.eval()

    KEY1_LS, KEY2_LS, CONF_LS = [], [], []
    total_pairs = len(tens_images) - 1
    
    # Use torch.cat if inputs are [1, 1, H, W], else stack
    use_cat = (len(tens_images[0].shape) == 4) 

    with torch.no_grad():
        for i in range(0, total_pairs, batch_size):
            curr_batch_end = min(i + batch_size, total_pairs)
            current_batch_size = curr_batch_end - i
            
            # --- 1. Prepare Batch ---
            raw_batch0 = tens_images[i : curr_batch_end]
            raw_batch1 = tens_images[i+1 : curr_batch_end+1]

            if use_cat:
                b_img0 = torch.cat(raw_batch0, dim=0)
                b_img1 = torch.cat(raw_batch1, dim=0)
            else:
                b_img0 = torch.stack(raw_batch0)
                b_img1 = torch.stack(raw_batch1)

            # --- 2. Pad (Crucial!) ---
            b_img0, (h, w) = pad_to_divisible(b_img0, 8)
            b_img1, _      = pad_to_divisible(b_img1, 8)

            # --- 3. Manual Split & Inference ---
            if num_gpus > 1 and current_batch_size >= 2:
                # Split batch in half
                mid = current_batch_size // 2
                
                # Chunk 1 -> GPU 0
                in0_a = b_img0[:mid].to('cuda:0')
                in1_a = b_img1[:mid].to('cuda:0')
                
                # Chunk 2 -> GPU 1
                in0_b = b_img0[mid:].to('cuda:1')
                in1_b = b_img1[mid:].to('cuda:1')

                # Parallel Inference
                # (PyTorch handles async execution automatically here)
                out_a = matcher0({"image0": in0_a, "image1": in1_a})
                out_b = matcher1({"image0": in0_b, "image1": in1_b})

                # Move results back to CPU immediately to merge
                # We merge lists of results
                res = {
                    'keypoints0': torch.cat([out_a['keypoints0'].cpu(), out_b['keypoints0'].cpu()]),
                    'keypoints1': torch.cat([out_a['keypoints1'].cpu(), out_b['keypoints1'].cpu()]),
                    'confidence': torch.cat([out_a['confidence'].cpu(), out_b['confidence'].cpu()]),
                    # Batch indexes for 'b' need to be offset by 'mid'
                    'batch_indexes': torch.cat([out_a['batch_indexes'].cpu(), out_b['batch_indexes'].cpu() + mid])
                }
            else:
                # Single GPU Fallback (or odd last batch)
                in0 = b_img0.to('cuda:0')
                in1 = b_img1.to('cuda:0')
                res = matcher0({"image0": in0, "image1": in1})
                # Move to CPU
                for k in res: res[k] = res[k].cpu()

            # --- 4. Process Results ---
            batch_ids = res['batch_indexes'].numpy()
            kpts0 = res['keypoints0'].numpy()
            kpts1 = res['keypoints1'].numpy()
            conf  = res['confidence'].numpy()

            for b in range(current_batch_size):
                mask = (batch_ids == b)
                kp0_b = kpts0[mask]
                kp1_b = kpts1[mask]
                conf_b = conf[mask]

                # Filter Padding
                if len(kp0_b) > 0:
                    valid_mask = (kp0_b[:, 0] < w) & (kp0_b[:, 1] < h) & \
                                 (kp1_b[:, 0] < w) & (kp1_b[:, 1] < h)
                    kp0_b = kp0_b[valid_mask]
                    kp1_b = kp1_b[valid_mask]
                    conf_b = conf_b[valid_mask]
                
                # Check 0 matches
                if len(kp0_b) == 0:
                    print(f"⚠️ Zero Matches for Pair {i+b}")
                    KEY1_LS.append(np.empty((0, 2)))
                    KEY2_LS.append(np.empty((0, 2)))
                    CONF_LS.append(np.empty((0,)))
                else:
                    KEY1_LS.append(kp0_b)
                    KEY2_LS.append(kp1_b)
                    CONF_LS.append(conf_b)

            if i % 20 == 0:
                print(f"Processed {i}/{total_pairs}")

    return KEY1_LS, KEY2_LS, CONF_LS

## Compute Relative Translation

In [ ]:
def relative_tran(Trans_arr):
    ls_rel_t = []
    for i in range(len(Trans_arr)-1):
        T1 = Trans_arr[i]
        T2 = Trans_arr[i+1]
        s_arr = np.array([0,0,0,1])
        T1 = np.vstack([T1,s_arr])
        T2 = np.vstack([T2,s_arr])
        #print(T1)
        #print(T2)
        T1_2 = np.dot(np.linalg.inv(T1), T2)
        
        ls_rel_t.append(T1_2[:3,:4])
        
    return ls_rel_t

## Camera Intrinsic Normalization and Stacked vectors

In [ ]:
def norm_and_stack_pairs(key1, key2, cam_K_inv):
    
    N, M, _ = key1.shape
    ones = np.ones((N, M, 1), dtype=key1.dtype)
    
    k1_h = np.concatenate([key1, ones], axis=2) 
    k2_h = np.concatenate([key2, ones], axis=2)

    k1_flat = k1_h.reshape(-1, 3)
    k2_flat = k2_h.reshape(-1, 3)
    
    k1_norm_flat = k1_flat @ cam_K_inv.T
    k2_norm_flat = k2_flat @ cam_K_inv.T
    
    k1_norm = k1_norm_flat.reshape(N, M, 3)
    k2_norm = k2_norm_flat.reshape(N, M, 3)
    
    L_Final = np.concatenate([k1_norm, k2_norm], axis=2)
    
    return L_Final

## Main code

In [ ]:
img_pth = 'kitti-images-file-path'
calib_pth = 'kitti-calib-file-path'

start = 0
end = len(natsorted(os.listdir(img_pth)))
skip = 1

count1 = []

poses_pth = 'kitti-poses-file-path'

trans_arr_ls = []

with open(poses_pth,'r') as file:
        for line in file:
            vals = list(map(float, line.strip().split()))
            grouped = [vals[i:i+4] for i in range(0, len(vals), 4)]
            trans_arr_ls.append(np.array(grouped))
print(f'Step1: Loaded {len(trans_arr_ls)} Raw Poses')

trans_arr_ls = trans_arr_ls[start:end:skip]

Rel_T = relative_tran(np.asarray(trans_arr_ls))

img_pts = natsorted(os.listdir(img_pth))

img_ls_pth = [os.path.join(img_pth,img_pts[i]) for i in range(len(img_pts))][start:end:skip]

_, tens_imgs = image_preprocess(img_ls_pth)

L_K1_Raw, L_K2_Raw, L_CONF_Raw = LOFTR_matcher_manual_split(tens_imgs, batch_size=8)

for i in range(len(L_K1_Raw)):
    c1 = len(L_K1_Raw[i])
    if c1 == 0:
        print(i)
    count1.append(c1)

print('The minimum matched points between images', min(count1))

if min(count1) == 0:
    thresh1 = 50
else:
    thresh1 = min(count1)

ACT_k1 = [L_K1_Raw[i][0:thresh1] for i in range(len(L_K1_Raw))]
ACT_k2 = [L_K2_Raw[i][0:thresh1] for i in range(len(L_K2_Raw))]

#R_arr_n = R_raw[start:end:skip]
#t_arr_n = t_raw[start:end:skip]

ACT_k1_arr = np.asarray(ACT_k1) 
ACT_k2_arr = np.asarray(ACT_k2)

K0 = extract_intrinsics(calib_pth, 'P2')
K0_inv = np.linalg.inv(K0)

# This will return shape (N, M, 6)
L_Final = norm_and_stack_pairs(ACT_k1_arr, ACT_k2_arr, K0_inv)

print(f"Final Tensor Shape: {L_Final.shape}") # Should be (N, M, 6)

In [ ]:
t_raw = np.array([Rel_T[i][:,3:4] for i in range(len(Rel_T))])
R_raw = np.array([Rel_T[i][:,:3] for i in range(len(Rel_T))])

## Saving the files

In [ ]:
ot_pt1 = 'Main-save-dir'
os.mkdir(ot_pt1)

In [ ]:
ot_pt2 = 'Main-save-sub-dir'

os.mkdir(ot_pt2)

np.save(ot_pt2+'/key_pts.npy',L_Final)
np.save(ot_pt2+'/Tran_arr.npy',t_raw)
np.save(ot_pt2+'/Rot_arr.npy',R_raw)